# Re-fetch & re-curate the clinical-stage GPCR ligand dataset

**Purpose:** independently reproduce the curated dataset straight from ChEMBL and verify the
headline counts (**3,011 records / 719 molecules / 47 targets / 18 families**), while printing a
**per-criterion filtration funnel** for full transparency.

The pipeline mirrors the thesis exactly:
binding (`Ki`) assays, *Homo sapiens*, valid pChEMBL value → parent structure (salt removal) →
MW < 700 Da → clinical Phase I–IV → peptide removal (≥ 2 amide bonds) → `pKi = pchembl_value` →
deduplicate on (molecule, target) keeping the highest pKi.

> **Reproducibility note.** ChEMBL is versioned. The thesis used **ChEMBL v36 (October 2025)**.
> Running against a *different* release will legitimately give a slightly different count — that is
> expected (the database grows), not a pipeline error. Pin the version when you report the number.

In [ ]:
# Run once. If you're on Colab, this is all you need; restart the runtime afterwards if prompted.
!pip -q install chembl_webresource_client rdkit pandas tqdm

In [ ]:
import math
import pandas as pd
from tqdm.auto import tqdm
from chembl_webresource_client.new_client import new_client
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem.SaltRemover import SaltRemover
from rdkit import RDLogger; RDLogger.DisableLog('rdApp.*')

pd.set_option('display.max_rows', 60)
print('ChEMBL client and RDKit ready.')

In [ ]:
PANEL = [
    # (target label, UniProt accession, family) — 47 retained targets
    ('Adenosine A1', 'P30542', 'Adenosine'),
    ('Adenosine A2A', 'P29274', 'Adenosine'),
    ('Adrenergic alpha1D', 'P25100', 'Adrenergic'),
    ('Adrenergic alpha2A', 'P08913', 'Adrenergic'),
    ('Adrenergic alpha2B', 'P18089', 'Adrenergic'),
    ('Adrenergic alpha2C', 'P18825', 'Adrenergic'),
    ('Adrenergic beta1', 'P08588', 'Adrenergic'),
    ('Adrenergic beta2', 'P07550', 'Adrenergic'),
    ('Adrenergic beta3', 'P13945', 'Adrenergic'),
    ('Angiotensin AT2', 'P50052', 'Angiotensin'),
    ('Bradykinin B2', 'P30411', 'Bradykinin'),
    ('Cannabinoid CB1', 'P21554', 'Cannabinoid'),
    ('Chemokine CCR2B', 'P41597', 'Chemokine'),
    ('Chemokine CCR4', 'P51679', 'Chemokine'),
    ('Chemokine CCR5', 'P51681', 'Chemokine'),
    ('Cholecystokinin CCKA', 'P32238', 'Cholecystokinin'),
    ('Dopamine D1', 'P21728', 'Dopamine'),
    ('Dopamine D2L', 'P14416', 'Dopamine'),
    ('Dopamine D3', 'P35462', 'Dopamine'),
    ('Dopamine D4.2', 'P21917', 'Dopamine'),
    ('Endothelin ETA', 'P25101', 'Endothelin'),
    ('Estrogen ERalpha', 'P03372', 'Estrogen'),
    ('Estrogen ERbeta', 'Q92731', 'Estrogen'),
    ('Melanocortin MC3', 'P32245', 'Melanocortin'),
    ('Melanocortin MC4', 'P32246', 'Melanocortin'),
    ('Melanocortin MC5', 'P33032', 'Melanocortin'),
    ('Muscarinic M1', 'P11229', 'Muscarinic'),
    ('Muscarinic M2', 'P08172', 'Muscarinic'),
    ('Muscarinic M3', 'P20309', 'Muscarinic'),
    ('Muscarinic M4', 'P08173', 'Muscarinic'),
    ('Muscarinic M5', 'P08912', 'Muscarinic'),
    ('Neuropeptide Y2', 'P49146', 'Neuropeptide Y'),
    ('Opiate delta', 'P41143', 'Opiate'),
    ('Opiate kappa', 'P41145', 'Opiate'),
    ('Opiate mu', 'P35372', 'Opiate'),
    ('Serotonin 5-HT1A', 'P08908', 'Serotonin'),
    ('Serotonin 5-HT1B', 'P28222', 'Serotonin'),
    ('Serotonin 5-HT2A', 'P28223', 'Serotonin'),
    ('Serotonin 5-HT2B', 'P41595', 'Serotonin'),
    ('Serotonin 5-HT2C', 'P28335', 'Serotonin'),
    ('Serotonin 5-HT3', 'P46098', 'Serotonin'),
    ('Serotonin 5-HT4', 'Q13639', 'Serotonin'),
    ('Serotonin 5-HT6', 'P50406', 'Serotonin'),
    ('Sigma1', 'Q99720', 'Sigma'),
    ('Tachykinin NK1', 'P25103', 'Tachykinin'),
    ('Tachykinin NK2', 'P21452', 'Tachykinin'),
    ('Vasopressin V1A', 'P37288', 'Vasopressin'),
    # --- 3 targets that drop out under the clinical-phase filter (expected: 0 compounds) ---
    ('Calcitonin', 'P30988', 'Calcitonin'),
    ('Neuropeptide Y1', 'P25929', 'Neuropeptide Y'),
    ('Leukotriene B4', 'Q15722', 'Leukotriene'),
]

print(f'{len(PANEL)} targets in panel (47 retained + 3 expected to drop).')

In [ ]:
# --- Map each UniProt accession to its human single-protein ChEMBL target id ---
target_api = new_client.target

def resolve_target(acc):
    hits = list(target_api.filter(target_components__accession=acc,
                                  target_type='SINGLE PROTEIN',
                                  organism='Homo sapiens').only(['target_chembl_id','pref_name']))
    return hits[0]['target_chembl_id'] if hits else None

target_ids = {}
for label, acc, fam in tqdm(PANEL, desc='resolving targets'):
    target_ids[label] = resolve_target(acc)
missing = [l for l,t in target_ids.items() if t is None]
print('unresolved:', missing or 'none')

In [ ]:
# --- Fetch Ki binding activities (human, with pChEMBL) for each target ---
activity_api = new_client.activity
FIELDS = ['molecule_chembl_id','canonical_smiles','standard_type','standard_value',
          'standard_units','pchembl_value','assay_type','target_chembl_id']

rows = []
for label, acc, fam in tqdm(PANEL, desc='fetching activities'):
    tid = target_ids.get(label)
    if not tid:
        continue
    acts = activity_api.filter(target_chembl_id=tid, standard_type='Ki', assay_type='B',
                               pchembl_value__isnull=False,
                               target_organism='Homo sapiens').only(FIELDS)
    for a in acts:
        a = dict(a); a['target'] = label; a['family'] = fam; a['uniprot'] = acc
        rows.append(a)

raw = pd.DataFrame(rows)
print('raw activity rows fetched:', len(raw))

In [ ]:
# --- Molecule-level max_phase (bulk) ---
molecule_api = new_client.molecule
ids = raw['molecule_chembl_id'].dropna().unique().tolist()
phase = {}
for i in tqdm(range(0, len(ids), 200), desc='max_phase'):
    chunk = ids[i:i+200]
    for m in molecule_api.filter(molecule_chembl_id__in=chunk).only(['molecule_chembl_id','max_phase']):
        phase[m['molecule_chembl_id']] = m['max_phase']
raw['max_phase'] = raw['molecule_chembl_id'].map(phase)

In [ ]:
# --- Curation pipeline with a per-step funnel ---
funnel = {}
def step(name, d):
    funnel[name] = (len(d), d['molecule_chembl_id'].nunique())
    return d

df = step('0. raw Ki / human / binding activities', raw.copy())

# 1. valid, numeric pChEMBL value
df['pchembl_value'] = pd.to_numeric(df['pchembl_value'], errors='coerce')
df = step('1. valid pChEMBL value', df.dropna(subset=['pchembl_value']))

# 2. parsable SMILES
df['mol'] = df['canonical_smiles'].apply(lambda s: Chem.MolFromSmiles(s) if isinstance(s, str) else None)
df = step('2. parsable SMILES', df[df['mol'].notna()])

# 3. salt removal -> parent structure
sr = SaltRemover()
def parent(m):
    p = sr.StripMol(m, dontRemoveEverything=True)
    if p is None or p.GetNumAtoms() == 0:
        frags = Chem.GetMolFrags(m, asMols=True)
        p = max(frags, key=lambda x: x.GetNumAtoms()) if frags else m
    return p
df['mol'] = df['mol'].apply(parent)
df['parent_smiles'] = df['mol'].apply(Chem.MolToSmiles)

# 4. molecular weight < 700 Da (RDKit, on parent)
df['MW'] = df['mol'].apply(Descriptors.MolWt)
df = step('3. MW < 700 Da', df[df['MW'] < 700])

# 5. clinical phase I-IV
df['max_phase'] = pd.to_numeric(df['max_phase'], errors='coerce')
df = step('4. clinical phase I-IV', df[df['max_phase'].isin([1,2,3,4])])

# 6. peptide removal: >= 2 amide-backbone bonds -> treated as chain, removed
amide = Chem.MolFromSmarts('C(=O)N')
df['n_amide'] = df['mol'].apply(lambda m: len(m.GetSubstructMatches(amide)))
df = step('5. peptide removed (< 2 amide bonds)', df[df['n_amide'] < 2])

# 7. pKi = pChEMBL value (per thesis); 8. dedup highest pKi per (molecule, target)
df['pKi'] = df['pchembl_value']
df = df.sort_values('pKi', ascending=False).drop_duplicates(['molecule_chembl_id','target'])
df = step('6. deduplicated (highest pKi / pair)', df)

In [ ]:
# --- Report ---
print('FILTRATION FUNNEL  (records | unique molecules)')
print('-'*56)
for name,(nr,nm) in funnel.items():
    print(f'  {nr:6d} | {nm:5d}   {name}')
print('-'*56)
print(f"FINAL   : {len(df):5d} records | {df['molecule_chembl_id'].nunique()} molecules | "
      f"{df['target'].nunique()} targets | {df['family'].nunique()} families")
print(f'EXPECTED :  3011 records |   719 molecules |  47 targets | 18 families  (ChEMBL v36)')

dropped = [t for t in ['Calcitonin','Neuropeptide Y1','Leukotriene B4'] if t not in set(df['target'])]
print('\nTargets that dropped out (expected 3):', dropped)

In [ ]:
# --- Save the re-fetched curated set ---
cols = ['molecule_chembl_id','target','family','uniprot','max_phase','pKi','MW','parent_smiles']
df[cols].to_csv('curated_refetch.csv', index=False)
print('wrote curated_refetch.csv  (', len(df), 'rows )')
df[cols].head()

### Interpreting the result

* If the final counts match **3,011 / 719 / 47 / 18**, the dataset is reproduced exactly.
* If they differ by a little, check the ChEMBL release first (`new_client` always hits the *latest*
  ChEMBL). A newer release than v36 will contain more `Ki` measurements, so the same filters retain
  a few more compounds — this is the expected explanation for a 2.6k → 3.0k type change and is not a
  pipeline error.
* `curated_refetch.csv` can be diff'd against `curated_15desc.csv` (on `molecule_chembl_id` + `target`)
  to see exactly which pairs were added or removed between releases.